In [ ]:
from pathlib import Path
import pprint

# run pip install -e . in the root directory to install this package
from stacbuilder import (
    build_collection,
    list_asset_metadata,
    list_input_files,
    list_stac_items,
    load_collection,
    validate_collection,
)

In [ ]:
# Collection configuration
catalog_version = "v05"
collection_config_path = Path("config-collection.json")

# Input Paths
tiff_input_path = Path("/data/MTDA/PEOPLE_EA/Landsat_three-annual_NDWI_v1")
tiffs_glob = "*/*.tif"

# Output Paths
output_path = Path("results")
test_output_path = output_path / "test" / catalog_version
publish_output_path = output_path / "publish" / catalog_version

# Openeo backend
openeo_backend_url = "https://openeo-dev.vito.be"

In [ ]:
# list input files
input_files = list_input_files(glob=tiffs_glob, input_dir=tiff_input_path, max_files=None)
print(f"Found {len(input_files)} input files. 5 first files:")
for i in input_files[:5]:
    print(i)

In [ ]:
# list meta data
asset_metadata = list_asset_metadata(
    collection_config_path=collection_config_path, glob=tiffs_glob, input_dir=tiff_input_path, max_files=1
)
for k in asset_metadata:
    pprint.pprint(k.to_dict())

In [ ]:
# list items
stac_items, failed_files = list_stac_items(
    collection_config_path=collection_config_path, glob=tiffs_glob, input_dir=tiff_input_path, max_files=1
)
print(f"Found {len(stac_items)} STAC items")
if failed_files:
    print(f"Failed files: {failed_files}")
if len(stac_items) > 0:
    print("First stac item:")
    stac_items[0]

In [ ]:
# build collection
build_collection(
    collection_config_path=collection_config_path,
    glob=tiffs_glob,
    input_dir=tiff_input_path,
    output_dir=test_output_path,
)
# TODO overrides defined in the collection config are no longer supported. This is easier done using pystac if needed for specific cases.
# In this case "overrides": {
# "extent/spatial/bbox": [
#     [
#     -58.6572887,
#     24.4650791,
#     39.6002888,
#     54.7014217
#     ]
# ],
# "properties/proj:epsg": 3035,
# "properties/proj:bbox": [
#     665000.0,
#     746700.0,
#     7332600.0,
#     5491700.0
#     ]
# } will have to be added manually to the collection.json file or by loading it in using load_collection underneath the build_collection call.

In [ ]:
# validate collection
validate_collection(
    collection_file=test_output_path / "collection.json",
)

In [ ]:
# show collection
load_collection(collection_file=test_output_path / "collection.json")